# SQL for Data Analysis Notebook

This notebook is the runnable practice alternative to the HTML SQL guide. Use it cell by cell to work through fundamentals, joins, window functions, analytical patterns, and finance queries.

Dataset files in this workspace:

- `sql-practice-users.csv`
- `sql-practice-orders.csv`
- `sql-practice-products.csv`
- `sql-practice-events.csv`
- `sql-practice-subscriptions.csv`
- `sql-finance-customers.csv`
- `sql-finance-invoices.csv`
- `sql-finance-payments.csv`
- `sql-finance-monthly-budget.csv`


## 1. Start Here

The notebook is organized from beginner to expert and includes both core SQL practice and finance analytics examples.


In [8]:

import sqlite3
from pathlib import Path

import pandas as pd

base_dir = Path(r"e:\vin\da")
conn = sqlite3.connect(":memory:")

sources = {
    "users": "sql-practice-users.csv",
    "orders": "sql-practice-orders.csv",
    "products": "sql-practice-products.csv",
    "events": "sql-practice-events.csv",
    "subscriptions": "sql-practice-subscriptions.csv",
    "customers": "sql-finance-customers.csv",
    "invoices": "sql-finance-invoices.csv",
    "payments": "sql-finance-payments.csv",
    "monthly_budget": "sql-finance-monthly-budget.csv",
}

for table_name, filename in sources.items():
    frame = pd.read_csv(base_dir / filename)
    frame.columns = [column.strip() for column in frame.columns]
    for column in frame.columns:
        if column.endswith("_date") or column in {"signup_date", "start_date", "end_date", "due_date", "month", "invoice_date", "payment_date", "launch_date"}:
            frame[column] = pd.to_datetime(frame[column], errors="coerce")
    frame.to_sql(table_name, conn, index=False, if_exists="replace")

def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

run_sql("""
SELECT COUNT(*) AS users_count FROM users;
""")

run_sql("""
SELECT COUNT(*) AS orders_count FROM orders;
""")

run_sql("""
SELECT COUNT(*) AS events_count FROM events;
""")

run_sql("""
SELECT COUNT(*) AS subscriptions_count FROM subscriptions;
""")

,subscriptions_count
0,10


## 2. Beginner SQL Fundamentals

Practice SELECT, FROM, WHERE, DISTINCT, LIMIT, and ORDER BY.


In [ ]:
run_sql (
    """
SELECT user_id, country, plan_type
FROM users
WHERE is_active = 1
ORDER BY signup_date DESC
LIMIT 10;
"""
) run_sql (
    """
SELECT DISTINCT country FROM users ORDER BY country;
"""
)

,country
0,Australia
1,Brazil
2,Canada
3,Germany
4,India
5,United Kingdom
6,United States


## 3. Filtering and Aggregation

Use WHERE before grouping and HAVING after grouping.


In [ ]:
run_sql (
    """
SELECT u.country, COUNT(o.order_id) AS order_count, SUM(
        o.quantity * o.unit_price - o.discount_amount
) AS net_revenue
FROM users u
    JOIN orders o ON u.user_id = o.user_id
WHERE
    o.order_status = 'completed'
GROUP BY
    u.country
HAVING
    SUM(
        o.quantity * o.unit_price - o.discount_amount
) > 100
ORDER BY net_revenue DESC;
"""
)

,country,order_count,net_revenue
0,United States,4,1041.0
1,Brazil,1,449.1
2,Canada,2,441.0
3,India,2,438.0
4,Germany,1,192.0
5,United Kingdom,1,120.0


## 4. Joins Practice

Use this section to understand row retention and one-to-many relationships.


In [ ]:
run_sql (
    """
SELECT u.user_id, u.country, o.order_id, o.order_date, o.quantity * o.unit_price - o.discount_amount AS net_revenue
FROM users u
    LEFT JOIN orders o ON u.user_id = o.user_id
ORDER BY u.user_id, o.order_date;
"""
)

,user_id,country,order_id,order_date,net_revenue
0,1,United States,1001,2024-06-01 00:00:00,230.0
1,1,United States,1002,2024-06-15 00:00:00,299.0
2,2,Canada,1003,2024-06-20 00:00:00,142.0
3,2,Canada,1012,2024-09-01 00:00:00,299.0
4,3,United Kingdom,1004,2024-07-05 00:00:00,120.0
5,4,India,1005,2024-07-08 00:00:00,93.0
6,5,United States,1006,2024-07-10 00:00:00,474.0
7,6,Australia,1007,2024-07-18 00:00:00,598.0
8,7,Germany,1008,2024-08-01 00:00:00,192.0
9,8,United States,1009,2024-08-03 00:00:00,38.0


## 5. Advanced Joins and Window Functions

Use window functions for ranking, running totals, and period comparisons.


In [ ]:
run_sql (
    """
WITH
    customer_revenue AS (
        SELECT u.country, o.user_id, SUM(
                o.quantity * o.unit_price - o.discount_amount
) AS revenue
        FROM orders o
            JOIN users u ON o.user_id = u.user_id
        WHERE
            o.order_status = 'completed'
        GROUP BY
            1,
            2
    )
SELECT *
FROM (
        SELECT country, user_id, revenue, ROW_NUMBER() OVER (
                PARTITION BY
                    country
                ORDER BY revenue DESC
            ) AS rn
        FROM customer_revenue
    ) ranked
WHERE
    rn <= 2
ORDER BY country, rn;
"""
)

,country,user_id,revenue,rn
0,Brazil,9,449.1,1
1,Canada,2,441.0,1
2,Germany,7,192.0,1
3,India,10,345.0,1
4,India,4,93.0,2
5,United Kingdom,3,120.0,1
6,United States,1,529.0,1
7,United States,5,474.0,2


In [ ]:
run_sql (
    """
WITH
    monthly AS (
        SELECT substr(order_date, 1, 7) || '-01' AS month, SUM(
                quantity * unit_price - discount_amount
) AS revenue
        FROM orders
        WHERE
            order_status = 'completed'
        GROUP BY
            1
    )
SELECT
    month,
    revenue,
    revenue - LAG(revenue) OVER (
        ORDER BY month
    ) AS mom_change,
    SUM(revenue) OVER (
        ORDER BY month
    ) AS running_revenue
FROM monthly
ORDER BY month;
"""
)

,month,revenue,mom_change,running_revenue
0,2024-06-01,671.0,NaN,671.0
1,2024-07-01,687.0,16.0,1358.0
2,2024-08-01,1024.1,337.1,2382.1
3,2024-09-01,299.0,-725.1,2681.1


## 6. Finance Practice

Apply the same analytical patterns to finance and collections use cases.


In [ ]:
run_sql (
    """
WITH
    invoice_rev AS (
        SELECT
            substr(i.invoice_date, 1, 7) || '-01' AS month,
            c.country,
            SUM(i.invoice_amount) AS invoiced_revenue
        FROM invoices i
            JOIN customers c ON i.customer_id = c.customer_id
        GROUP BY
            1,
            2
    )
SELECT r.month, r.country, r.invoiced_revenue, b.revenue_target, r.invoiced_revenue - b.revenue_target AS variance_to_target
FROM
    invoice_rev r
    LEFT JOIN monthly_budget b ON r.month = b.month
    AND r.country = b.country
ORDER BY r.month, r.country;
"""
)

,month,country,invoiced_revenue,revenue_target,variance_to_target
0,2024-06-01,Australia,9800,None,None
1,2024-06-01,Canada,7200,None,None
2,2024-06-01,Germany,31000,None,None
3,2024-06-01,India,5600,None,None
4,2024-06-01,United Kingdom,13400,None,None
5,2024-06-01,United States,25000,None,None
6,2024-07-01,Brazil,4500,None,None
7,2024-07-01,Canada,7600,None,None
8,2024-07-01,India,6100,None,None
9,2024-07-01,United Kingdom,12000,None,None


## 7. Project Prompts

Use these as notebook exercises or interview practice tasks.


1. Build a monthly revenue dashboard query from the sales tables.
2. Find the top 2 customers per country by revenue.
3. Calculate collection rate by month and country.
4. Create a churn flag output from the subscriptions table.


## 8. Interview Notes

Remember the main query order: SELECT -> FROM -> JOIN -> WHERE -> GROUP BY -> HAVING -> WINDOW -> ORDER BY.
